# Data loading - Eurostat

In [2]:
import requests
import gzip
import pandas as pd
import io
from pathlib import Path
import emoji

In [55]:
# Eurostat's SDMX 2.1 bulk download
# {code} is filled in per dataset; compressed=true means the response body is gzip-encoded TSV.

BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{code}?format=TSV&compressed=true"

# Dataset codes to download, mapped to a human-readable label (for your own reference,
# not used by the code below).
eurostat_datasets={
    'nama_10r_2gdp': 'GDP',
    'lfst_r_lfu3rt': 'unemployment_rate',
    'isoc_r_iuse_i': 'internet_usage',
    'ilc_li41': 'poverty_risk',
    'tps00208': 'life_expectancy',
    'tps00100': 'social_expenditure',
    'hlth_ehis_fv3e': 'veggie_consumption',
    'hlth_ehis_pe2e': 'aerobic_activity',
    'hlth_ehis_al3e': 'heavy_drinking',
    'hlth_ehis_sk3e': 'smoking',
    'sdg_02_10': 'obesity_rate',
    'demo_r_mlifexp': 'life_satisfaction',
    'ilc_scp09': 'family_contact',
    'ilc_scp19': 'voluntary_participation',
    'ilc_scp15': 'social_support',
    'ilc_scp17': 'social_contact',
    'ilc_pw03': 'social_trust',
}

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(exist_ok=True)


def download_and_save(code, label):
    """
    Download a single Eurostat dataset by code and save it as a plain-text
    .tsv file in RAW_DIR, named after its human-readable label rather than
    its raw code. Skips the download if the file already exists.
    """
    save_path = RAW_DIR / f'{eurostat_datasets[code]}.tsv'

    # Don't re-download files that are already on disk.
    if save_path.exists():
        print(f'⏭️ SKIP {eurostat_datasets[code]} -> already exists')
        return save_path, False

    # Request URL for dataset code and fetch it.
    url = BASE_URL.format(code=code)
    r = requests.get(url, timeout=120)
    r.raise_for_status() # raises an exception on HTTP error status codes (404, 500, etc.)

    # Checking to have gzip data, not HTML
    content = r.content
    if content[:2] != b"\x1f\x8b":  # gzip magic number
        raise ValueError(f"[{code}] response is not gzip data (got {content[:30]!r})")

    # Write the decompressed TSV content to disk under the human-readable filename.
    text = gzip.decompress(content).decode('utf-8')

    save_path.write_text(text, encoding="utf-8")
    return save_path, True

In [56]:
paths = {}

for code, label in eurostat_datasets.items():
    try:
        path, downloaded = download_and_save(code, label)
        if downloaded:
            print(f'✅ OK: {code} -> {path}')
        paths[code] = path
    except Exception as e:
        print(f'⛔ FAIL {code}: {e}')
        paths[code] = None

✅ OK: nama_10r_2gdp -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/GDP.tsv
✅ OK: lfst_r_lfu3rt -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/unemployment_rate.tsv
✅ OK: isoc_r_iuse_i -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/internet_usage.tsv
✅ OK: ilc_li41 -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/poverty_risk.tsv
✅ OK: tps00208 -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/life_expectancy.tsv
✅ OK: tps00100 -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/social_expenditure.tsv
✅ OK: hlth_ehis_fv3e -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/veggie_consumption.tsv
✅ OK: hlth_ehis_pe2e -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/aerobic_activity.tsv
✅ OK: hlth_ehis_al3e -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/heavy_drinking.tsv
✅ OK: hlth_ehis_sk3e -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/smoking.tsv
✅ OK: sdg_02_10 -> /home/zbezjak/code/ZanBezjak/bluezone-tool/data/raw/obesit

In [57]:
def load_dataset(code):
    """
    Load a previously downloaded Eurostat .tsv file into a raw DataFrame.
    No cleaning/reshaping here — just parsing, so you can inspect the
    actual raw structure Eurostat gives you before deciding how to tidy it.
    """
    path = RAW_DIR / f'{eurostat_datasets[code]}.tsv'
    df = pd.read_csv(path, sep='\t', na_values=[':', ': '])
    return df

# Load every dataset you successfully downloaded into a dict: {code: raw_dataframe}
dfs = {label: load_dataset(code) for code, label in eurostat_datasets.items()}

In [58]:
dfs['GDP'].head()

,"freq,unit,geo\TIME_PERIOD",2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"A,EUR_HAB,AL",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,...,3600,3800,4100,4500,4900,4700,5400,6500,NaN,NaN
1,"A,EUR_HAB,AL0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,...,3600,3800,4100,4500,4900,4700,5400,6500,NaN,NaN
2,"A,EUR_HAB,AL01",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2400,...,2900,3100,3300,3600,3900,3600,4300,5300,NaN,NaN
3,"A,EUR_HAB,AL02",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4200,...,4300,4500,5000,5500,5900,5800,6500,7700,NaN,NaN
4,"A,EUR_HAB,AL03",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500,...,3300,3500,3700,4200,4500,4200,5000,6000,NaN,NaN


In [23]:
def parse(df):
    # Split first column into individual columns
    first_col = df.columns[0]
    dims = first_col.split(",")
    df[dims] = df[first_col].str.split(",", expand=True)
    df = df.drop(columns=[first_col])

    # Clean geo codes (strip whitespace)
    df["geo\TIME_PERIOD"] = df["geo\TIME_PERIOD"].str.strip()
    df = df.rename(columns={"geo\TIME_PERIOD": "geo"})

    return df

<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:10: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:10: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_172949/1921598825.py:9: SyntaxWarning: invalid escape sequence '\T'
  df["geo\TIME_PERIOD"] = df["geo\TIME_PERIOD"].str.strip()
/tmp/ipykernel_172949/1921598825.py:9: SyntaxWarning: invalid escape sequence '\T'
  df["geo\TIME_PERIOD"] = df["geo\TIME_PERIOD"].str.strip()
/tmp/ipykernel_172949/1921598825.py:10: SyntaxWarning: invalid escape sequence '\T'
  df = df.rename(columns={"geo\TIME_PERIOD": "geo"})


In [35]:
parsed_gdp = parse(dfs['GDP'])
parsed_gdp.head()

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2018,2019,2020,2021,2022,2023,2024,freq,unit,geo
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,3000,...,4500,4900,4700,5400,6500,NaN,NaN,A,EUR_HAB,AL
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,3000,...,4500,4900,4700,5400,6500,NaN,NaN,A,EUR_HAB,AL0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2400,2400,...,3600,3900,3600,4300,5300,NaN,NaN,A,EUR_HAB,AL01
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4200,4000,...,5500,5900,5800,6500,7700,NaN,NaN,A,EUR_HAB,AL02
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500,2500,...,4200,4500,4200,5000,6000,NaN,NaN,A,EUR_HAB,AL03


In [36]:
parsed_gdp['geo'].str.len().value_counts()

geo
4    2067
3     821
2     234
9       7
Name: count, dtype: int64

In [28]:
parsed_gdp['unit'].unique()

array(['EUR_HAB', 'EUR_HAB_EU27_2020', 'MIO_EUR', 'MIO_NAC',
       'MIO_PPS_EU27_2020', 'PPS_EU27_2020_HAB', 'PPS_HAB_EU27_2020'],
      dtype=object)

In [30]:
parsed_gdp['freq'].unique()

array(['A'], dtype=object)

In [31]:
all_geos = parsed_gdp["geo"].unique()
aggregates = [g for g in all_geos if not (2 <= len(g) <= 4 and g[:2].isalpha() and g[:2].isupper())]
print(sorted(aggregates))

['EU27_2020']


In [33]:
long_codes = parsed_gdp[parsed_gdp["geo"].str.len() > 4]["geo"].unique()
print(sorted(long_codes))

['EU27_2020']


In [38]:
parsed_gdp[parsed_gdp['geo'] == 'EU27_2020']

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2018,2019,2020,2021,2022,2023,2024,freq,unit,geo
165,18400.0,19300,19900.0,20400.0,21300.0,22100.0,23300.0,24700.0,25400,24200,...,30600,31700,30500,33300,36100,38400,39900,A,EUR_HAB,EU27_2020
594,100.0,100,100.0,100.0,100.0,100.0,100.0,100.0,100,100,...,100,100,100,100,100,100,100,A,EUR_HAB_EU27_2020,EU27_2020
1040,7884970.7,8260284.9,8559614.0,8792709.3,9195744.5,9600083.4,10160793.1,10788346.2,11136839.3,10650843.1,...,13628646.5,14122772.4,13578815.3,14792278.1,16169785.3,17257484,18016189.9,A,MIO_EUR,EU27_2020
1511,7884970.7,8260284.9,8559614.0,8792709.3,9195744.5,9600083.4,10160793.1,10788346.2,11136839.3,10650843.1,...,13628646.5,14122772.4,13578815.3,14792278.1,16169785.3,17257484,18016189.9,A,MIO_NAC,EU27_2020
1982,7884970.7,8260284.9,8559614.0,8792709.3,9195744.5,9600083.4,10160793.1,10788346.2,11136839.3,10650843.1,...,13628646.5,14122772.4,13578815.3,14792278.1,16169785.3,17257484,18016189.9,A,MIO_PPS_EU27_2020,EU27_2020
2436,18400.0,19300,19900.0,20400.0,21300.0,22100.0,23300.0,24700.0,25400,24200,...,30600,31700,30500,33300,36100,38400,39900,A,PPS_EU27_2020_HAB,EU27_2020
2865,100.0,100,100.0,100.0,100.0,100.0,100.0,100.0,100,100,...,100,100,100,100,100,100,100,A,PPS_HAB_EU27_2020,EU27_2020


In [71]:
DATASETS = {
    "aerobic_activity": {
        "code":    "hlth_ehis_pe2e",
        "filters": {"unit": "PC", "sex": "T", "age": "TOTAL",
                    "isced11": "TOTAL", "duration": "MN_GE150"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% meeting 150+ min/week. 2014 and 2019 only.",
    },
    "family_contact": {
        "code":    "ilc_scp09",
        "filters": {"unit": "PC", "sex": "T", "age": "Y_GE16",
                    "isced11": "TOTAL", "frequenc": "WEEK",
                    "pers_cat": "FAM_REL"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% getting together with family at least weekly. 2015 only.",
    },
    "gdp_per_capita": {
        "code":    "nama_10r_2gdp",
        "filters": {"unit": "PPS_HAB_EU27_2020"},
        "level":   "all",
        "keep_aggregates": True,
        "notes":   "GDP per capita in PPS. NUTS0-2 confirmed.",
    },
    "heavy_drinking": {
        "code":    "hlth_ehis_al3e",
        "filters": {"unit": "PC", "sex": "T", "age": "TOTAL",
                    "isced11": "TOTAL", "frequenc": "GE1W"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% drinking heavily at least once/week. 2014 and 2019 only.",
    },
    "internet_usage": {
        "code":    "isoc_r_iuse_i",
        "filters": {"unit": "PC_IND", "indic_is": "I_IUSE"},
        "level":   "all",
        "keep_aggregates": True,
        "notes":   "% regularly using internet. NUTS2 confirmed.",
    },
    "life_expectancy": {
        "code":    "demo_r_mlifexp",
        "filters": {"unit": "YR", "sex": "F",   # ⚠ no T — only F and M available
                    "hlth_hle": "LE_Y0"},         # LE_Y0 = at birth
        "level":   "all",
        "keep_aggregates": True,
        "notes":   "Life expectancy at birth. No total sex — use F and M separately then average.",
    },
    "life_satisfaction": {
        "code":    "ilc_pw01b",
        "filters": {"unit": "YR", "sex": "T",    # ⚠ unit is YR not RTG
                    "age": "Y_LT1"},              # age dimension holds the score age here
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "⚠ This looks like life years not satisfaction — verify after download.",
    },
    "obesity_rate": {
        "code":    "sdg_02_10",
        "filters": {"unit": "PC", "bmi": "BMI_GE30"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% population BMI>=30.",
    },
    "poverty_risk": {
        "code":    "ilc_li41",
        "filters": {"unit": "PC"},
        "level":   "all",
        "keep_aggregates": True,
        "notes":   "No sub-dimensions — just filter unit. NUTS2 confirmed.",
    },
    "smoking": {
        "code":    "hlth_ehis_sk3e",
        "filters": {"unit": "PC", "sex": "T", "age": "TOTAL",
                    "isced11": "TOTAL", "smoking": "TOTAL"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% smokers total (light + heavy). 2014 and 2019 only.",
    },
    "social_contact": {
        "code":    "ilc_scp17",
        "filters": {"unit": "PC", "sex": "T", "age": "Y_GE16",
                    "isced11": "TOTAL"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% with no one to discuss personal matters. 2015 only.",
    },
    "social_expenditure": {
        "code":    "tps00100",
        "filters": {"unit": "PPS_HAB", "spdeps": "TOTAL"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "Social protection expenditure per inhabitant in PPS.",
    },
    "social_support": {
        "code":    "ilc_scp15",
        "filters": {"unit": "PC", "sex": "T", "age": "Y_GE16",
                    "isced11": "TOTAL"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "% with no one to ask for help. 2015 only.",
    },
    "social_trust": {
        "code":    "ilc_pw03",
        "filters": {"unit": "RTG", "sex": "T", "age": "Y_GE16",
                    "isced11": "TOTAL", "statinfo": "AVG",
                    "domain": "OTH"},
        "level":   "country",
        "keep_aggregates": True,
        "notes":   "Mean trust rating. 2013, 2018, 2022 only.",
    },
    "unemployment_rate": {
        "code":    "lfst_r_lfu3rt",
        "filters": {"unit": "PC", "sex": "T", "age": "Y_GE15",
                    "isced11": "TOTAL"},
        "level":   "all",
        "keep_aggregates": True,
        "notes":   "Total unemployment rate. NUTS2 confirmed.",
    },
}

In [46]:
def download(code):
    """Download dataset and return as raw dataframe."""
    save_path, _ = download_and_save(code, eurostat_datasets[code])
    df = pd.read_csv(save_path, sep="\t", na_values=[":", ": "])
    return df

In [48]:
for name, config in DATASETS.items():
    raw = download(config["code"])
    df = parse(raw)
    non_year = [c for c in df.columns if not str(c).strip().lstrip('-').isdigit()]
    print(f"\n{name}: {non_year}")

⏭️ SKIP aerobic_activity -> already exists

aerobic_activity: ['freq', 'unit', 'duration', 'isced11', 'sex', 'age', 'geo']
⏭️ SKIP family_contact -> already exists

family_contact: ['freq', 'frequenc', 'pers_cat', 'isced11', 'age', 'sex', 'unit', 'geo']
⏭️ SKIP GDP -> already exists

GPD: ['freq', 'unit', 'geo']
⏭️ SKIP heavy_drinking -> already exists

heavy_drinking: ['freq', 'unit', 'frequenc', 'isced11', 'sex', 'age', 'geo']
⏭️ SKIP internet_usage -> already exists

internet_usage: ['freq', 'indic_is', 'unit', 'geo']
⏭️ SKIP life_expectancy -> already exists

life_expectancy: ['freq', 'unit', 'sex', 'hlth_hle', 'geo']
⏭️ SKIP life_satisfaction -> already exists

life_satisfaction: ['freq', 'life_sat', 'statinfo', 'isced11', 'sex', 'age', 'unit', 'geo']
⏭️ SKIP obesity_rate -> already exists

obesity_rate: ['freq', 'unit', 'bmi', 'geo']
⏭️ SKIP poverty_risk -> already exists

poverty_risk: ['freq', 'unit', 'geo']
⏭️ SKIP smoking -> already exists

smoking: ['freq', 'unit', 'smoking'

In [60]:
eurostat_datasets = {config["code"]: name for name, config in DATASETS.items()}


In [ ]:
with open("../data/raw/dataset_dimensions.txt", "w") as f:
    for name, config in DATASETS.items():
        code = config["code"]
        df = download(code)
        df = parse(df)
        filter_cols = [c for c in df.columns
                       if c not in ["freq", "geo"]
                       and not str(c).strip().lstrip('-').isdigit()]
        f.write(f"\n── {name} ({code}) ──\n")
        for col in filter_cols:
            f.write(f"  {col}: {sorted(df[col].dropna().unique().tolist())}\n")
        f.write("\n")

print("Done — open data/raw/dataset_dimensions.txt")

⏭️ SKIP aerobic_activity -> already exists
⏭️ SKIP family_contact -> already exists
⏭️ SKIP gdp_per_capita -> already exists
⏭️ SKIP heavy_drinking -> already exists
⏭️ SKIP internet_usage -> already exists
⏭️ SKIP life_expectancy -> already exists
⏭️ SKIP life_satisfaction -> already exists
⏭️ SKIP obesity_rate -> already exists
⏭️ SKIP poverty_risk -> already exists
⏭️ SKIP smoking -> already exists
⏭️ SKIP social_contact -> already exists
⏭️ SKIP social_expenditure -> already exists
⏭️ SKIP social_support -> already exists
⏭️ SKIP social_trust -> already exists
⏭️ SKIP unemployment_rate -> already exists
Done — open data/raw/dataset_dimensions.txt


In [70]:
df = parse(dfs['life_expectancy'])
print(df["sex"].unique())

['F' 'M']


In [74]:
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(exist_ok=True)

SUPRANATIONAL = ["EU", "EU27_2020", "EU28", "EA", "EA19", "EA20", "EEA", "EEA31"]

# ── Collector ─────────────────────────────────────────────────────────────────
report = []

def to_wide(df):
    year_cols = [c for c in df.columns if str(c).strip().lstrip('-').isdigit()]
    df = df[["geo"] + year_cols].copy()
    df.columns = ["geo"] + [int(c) for c in year_cols]
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.extract(r"([\d.]+)", expand=False),
            errors="coerce"
        )
    return df.set_index("geo").sort_index()

def filter_geo(df, level, keep_aggregates):
    if not keep_aggregates:
        df = df[~df.index.isin(SUPRANATIONAL)]
    if level == "nuts2":
        return df[df.index.str.len() == 4]
    elif level == "country":
        return df[df.index.str.len() == 2]
    elif level == "all":
        return df[df.index.str.len().isin([2, 3, 4])]
    return df

def process_dataset(name, config):
    print(f"── {name} ({config['code']})")

    df = download(config["code"])
    df = parse(df)

    if name == "life_expectancy":
        for col, val in config["filters"].items():
            if col != "sex" and col in df.columns:
                df = df[df[col] == val]
        year_cols = [c for c in df.columns if str(c).strip().lstrip('-').isdigit()]
        for col in year_cols:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.extract(r"([\d.]+)", expand=False),
                errors="coerce"
            )
        df_f = df[df["sex"] == "F"][["geo"] + year_cols].set_index("geo")
        df_m = df[df["sex"] == "M"][["geo"] + year_cols].set_index("geo")
        df = ((df_f + df_m) / 2).reset_index()
    else:
        for col, val in config["filters"].items():
            if col in df.columns:
                df = df[df[col] == val]
            else:
                print(f"  ⚠ '{col}' not found — skipping")

    df = to_wide(df)
    df = filter_geo(df, config["level"], config["keep_aggregates"])

    # Collect report stats
    level_counts = df.index.str.len().value_counts().sort_index().to_dict()
    missing_pct  = df.isnull().sum().sum() / df.size * 100
    report.append({
        "dataset":       name,
        "code":          config["code"],
        "level":         config["level"],
        "nuts0":         level_counts.get(2, 0),
        "nuts1":         level_counts.get(3, 0),
        "nuts2":         level_counts.get(4, 0),
        "year_min":      int(min(df.columns)),
        "year_max":      int(max(df.columns)),
        "n_years":       len(df.columns),
        "shape":         str(df.shape),
        "missing_pct":   round(missing_pct, 1),
        "notes":         config.get("notes", ""),
    })

    path = INTERIM_DIR / f"{name}.parquet"
    df.to_parquet(path)
    print(f"  ✓ {df.shape}  |  {min(df.columns)}–{max(df.columns)}  |  missing: {missing_pct:.1f}%")

    return df

# ── Run all ───────────────────────────────────────────────────────────────────
interim = {}
for name, config in DATASETS.items():
    interim[name] = process_dataset(name, config)

# ── Summary report ────────────────────────────────────────────────────────────
report_df = pd.DataFrame(report)
print("\n")
print(report_df.to_string(index=False))

# Save report too
report_df.to_csv(INTERIM_DIR / "collection_report.csv", index=False)
print(f"\n✅ Report saved → data/interim/collection_report.csv")

── aerobic_activity (hlth_ehis_pe2e)
⏭️ SKIP aerobic_activity -> already exists
  ✓ (32, 2)  |  2014–2019  |  missing: 7.8%
── family_contact (ilc_scp09)
⏭️ SKIP family_contact -> already exists
  ✓ (36, 2)  |  2015–2022  |  missing: 8.3%
── gdp_per_capita (nama_10r_2gdp)
⏭️ SKIP gdp_per_capita -> already exists
  ✓ (428, 25)  |  2000–2024  |  missing: 3.1%
── heavy_drinking (hlth_ehis_al3e)
⏭️ SKIP heavy_drinking -> already exists
  ✓ (32, 2)  |  2014–2019  |  missing: 7.8%
── internet_usage (isoc_r_iuse_i)
⏭️ SKIP internet_usage -> already exists
  ✓ (391, 20)  |  2006–2025  |  missing: 25.1%
── life_expectancy (demo_r_mlifexp)
⏭️ SKIP life_expectancy -> already exists
  ✓ (31, 12)  |  2013–2024  |  missing: 2.4%
── life_satisfaction (ilc_pw01b)
⏭️ SKIP life_satisfaction -> already exists
  ✓ (37, 35)  |  1990–2024  |  missing: 9.6%
── obesity_rate (sdg_02_10)
⏭️ SKIP obesity_rate -> already exists
  ✓ (37, 6)  |  2008–2025  |  missing: 19.8%
── poverty_risk (ilc_li41)
⏭️ SKIP povert

In [77]:
report = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'collection_report.csv')
report.drop(columns='notes')

,dataset,code,level,nuts0,nuts1,nuts2,year_min,year_max,n_years,shape,missing_pct
0,aerobic_activity,hlth_ehis_pe2e,country,32,0,0,2014,2019,2,"(32, 2)",7.8
1,family_contact,ilc_scp09,country,36,0,0,2015,2022,2,"(36, 2)",8.3
2,gdp_per_capita,nama_10r_2gdp,all,33,110,285,2000,2024,25,"(428, 25)",3.1
3,heavy_drinking,hlth_ehis_al3e,country,32,0,0,2014,2019,2,"(32, 2)",7.8
4,internet_usage,isoc_r_iuse_i,all,38,127,226,2006,2025,20,"(391, 20)",25.1
5,life_expectancy,demo_r_mlifexp,all,31,0,0,2013,2024,12,"(31, 12)",2.4
6,life_satisfaction,ilc_pw01b,country,37,0,0,1990,2024,35,"(37, 35)",9.6
7,obesity_rate,sdg_02_10,country,37,0,0,2008,2025,6,"(37, 6)",19.8
8,poverty_risk,ilc_li41,all,38,100,283,2003,2025,23,"(421, 23)",44.8
9,smoking,hlth_ehis_sk3e,country,32,0,0,2014,2019,2,"(32, 2)",3.1


In [79]:
df = download("demo_r_mlifexp")
df = parse(df)
print("hlth_hle:", df["hlth_hle"].unique())
print("geo lengths:", df["geo"].str.len().value_counts().sort_index())

⏭️ SKIP life_expectancy -> already exists
hlth_hle: ['LE_Y0']
geo lengths: geo
2    62
9     2
Name: count, dtype: int64


In [80]:
raw_path = PROJECT_ROOT / "data" / "raw" / "life_expectancy.tsv"
raw = pd.read_csv(raw_path, sep="\t", nrows=5)
print(raw.head())
print(raw.columns.tolist())

  freq,unit,sex,hlth_hle,geo\TIME_PERIOD  2013   2014   2015   2016   2017   \
0                        A,YR,F,LE_Y0,AT   83.8   84.0   83.7   84.1   84.0   
1                        A,YR,F,LE_Y0,BE   83.2   83.9   83.4   84.0   83.9   
2                        A,YR,F,LE_Y0,BG   78.6   78.0   78.2   78.5   78.4   
3                        A,YR,F,LE_Y0,CH   85.0   85.4   85.1   85.6   85.6   
4                        A,YR,F,LE_Y0,CY   85.0   84.3   83.7   84.9   84.2   

   2018   2019   2020   2021   2022   2023   2024   
0   84.1   84.2   83.6   83.7   83.6   84.2   84.3  
1   83.9   84.3   83.0   84.3   83.9   84.5   84.4  
2   78.6   78.8   77.5   75.1   77.9   79.7   79.5  
3   85.7   85.8   85.1   85.8   85.5   86.0   85.9  
4   84.8   84.4   84.4   83.4   83.4   85.3   85.6  
['freq,unit,sex,hlth_hle,geo\\TIME_PERIOD', '2013 ', '2014 ', '2015 ', '2016 ', '2017 ', '2018 ', '2019 ', '2020 ', '2021 ', '2022 ', '2023 ', '2024 ']
